In [87]:
import joblib
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import NearestNeighbors


In [63]:
df = pd.read_csv("../data/raw/internal-link (7).csv")

df

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [65]:
df.isnull().sum()


age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

## Limpieza de Datos

In [66]:
df.replace("?", np.nan, inplace=True)
df.dropna(inplace=True)

Elimino registros con datos mal codificados o incompletos

## Target

> - Convertimos el ingreso en variable binaria (1 = gana más de 50K).

In [67]:
df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

df["income"].value_counts()

income
0    22654
1     7508
Name: count, dtype: int64

## SPLIT

In [68]:
X = df.drop("income", axis=1)
y = df["income"]

### Separo variables predictoras y variable objetivo

In [69]:
num_cols = X.select_dtypes(include=["int64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

In [70]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

## Modelo de Clasificación

In [71]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [72]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [73]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.92      0.90      4533
           1       0.72      0.60      0.65      1500

    accuracy                           0.84      6033
   macro avg       0.80      0.76      0.78      6033
weighted avg       0.84      0.84      0.84      6033



## Importancia de variables

In [74]:
feature_names = (
    list(num_cols) +
    list(model.named_steps["preprocessor"]
         .named_transformers_["cat"]
         .get_feature_names_out(cat_cols))
)

coefficients = model.named_steps["classifier"].coef_[0]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": coefficients
}).sort_values(by="importance", ascending=False)

importance_df.head(10)

,feature,importance
3,capital.gain,2.320807
30,marital.status_Married-AF-spouse,1.458973
55,relationship_Wife,1.227655
31,marital.status_Married-civ-spouse,1.095830
63,native.country_Cambodia,0.874227
39,occupation_Exec-managerial,0.822569
84,native.country_Italy,0.782124
2,education.num,0.731949
48,occupation_Tech-support,0.726823
46,occupation_Protective-serv,0.724587


## Definición del sistema de recomendación
> - ¿Qué queremos recomendar?
>
Trayectorias educativas y ocupacionales que aumenten probabilidad de >50K.
>
> - ¿Quién es el usuario?
>
Una persona adulta con perfil socioeconómico específico.
> 
> - ¿Qué define el perfil?
>
Edad, educación, ocupación, estado civil, horas trabajadas, etc.

## Sistema basado en contenido (k-NN)

> - Representamos cada persona como vector numérico y medimos similitud.

In [75]:
X_processed = model.named_steps["preprocessor"].transform(X)


nn = NearestNeighbors(n_neighbors=5, metric="cosine")
nn.fit(X_processed)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [76]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'income'],
      dtype='object')

In [77]:
perfil_usuario = pd.DataFrame([{
    "age": 25,
    "workclass": "Private",
    "fnlwgt": 200000,              # valor simulado promedio
    "education": "HS-grad",
    "education.num": 9,
    "marital.status": "Never-married",
    "occupation": "Sales",
    "relationship": "Not-in-family",
    "race": "White",
    "sex": "Male",
    "capital.gain": 0,
    "capital.loss": 0,
    "hours.per.week": 20,
    "native.country": "United-States"
}])


In [78]:
usuario_processed = model.named_steps["preprocessor"].transform(perfil_usuario)

Probabilidad de ganar >50K

In [79]:
probabilidad = model.predict_proba(perfil_usuario)[0][1]

print(f"Probabilidad estimada de ganar >50K: {probabilidad:.2%}")


Probabilidad estimada de ganar >50K: 1.75%


Convertimos todos los usuarios en vectores numéricos para medir similitud.

In [80]:
X_processed = model.named_steps["preprocessor"].transform(X)

Modelo k-NN

In [81]:
nn = NearestNeighbors(n_neighbors=5, metric="cosine")
nn.fit(X_processed)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [82]:
usuario_processed = model.named_steps["preprocessor"].transform(perfil_usuario)

Convertimos el perfil simulado al mismo formato vectorial.

In [83]:
distances, indices = nn.kneighbors(usuario_processed)

vecinos = df.iloc[indices[0]]
vecinos

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
30219,24,Private,229826,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,30,United-States,0
8663,20,Private,216972,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,30,United-States,0
11526,24,Private,153082,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,30,United-States,0
11437,18,Private,181755,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,30,United-States,0
20238,35,Private,140752,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,25,United-States,0


## Filtro vecinos exitosos

In [84]:
nn = NearestNeighbors(n_neighbors=50, metric="cosine")
nn.fit(X_processed)

distances, indices = nn.kneighbors(usuario_processed)

vecinos = df.iloc[indices[0]]

vecinos_exitosos = vecinos[vecinos["income"] == 1]

vecinos_exitosos.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income


"Con este perfil, no existen trayectorias similares exitosas. Se requieren cambios estructurales."

Comparar con TODOS los >50K

In [85]:
exitosos = df[df["income"] == 1]

exitosos[["education", "occupation", "hours.per.week"]].describe()


,hours.per.week
count,7508.000000
mean,45.706580
std,10.736987
min,1.000000
25%,40.000000
50%,40.000000
75%,50.000000
max,99.000000


In [86]:
perfil_modificado = perfil_usuario.copy()
perfil_modificado["hours.per.week"] = 40

nueva_probabilidad = model.predict_proba(perfil_modificado)[0][1]

print("Probabilidad original:", f"{probabilidad:.2%}")
print("Probabilidad trabajando 40h:", f"{nueva_probabilidad:.2%}")

Probabilidad original: 1.75%
Probabilidad trabajando 40h: 3.09%


In [88]:
joblib.dump(model, "../models/adult_income_model.pkl")

print("Modelo guardado correctamente en la carpeta models.")

Modelo guardado correctamente en la carpeta models.
